# Lezione 6: Il Campo Vettoriale su Grafo (Graph Neural Network)
**File**: [`models/graph_vector_field.py`](file:///g:/My%20Drive/PhD%20&%20Research/Temporal%20GNN/cfm_grn/models/graph_vector_field.py)  
**Ruolo nel progetto**: È l'architettura neurale centrale che stima la velocità di variazione di ciascun gene $v_{\theta, i}(x, t, c)$ propagando informazioni lungo i circuiti regolatori orientati.


---
## 1. Il Principio Biologico del Message Passing Orientato

Nelle cellule viventi, la velocità di trascrizione istantanea del gene bersaglio $i$:
$$\frac{dx_i}{dt} = v_{\theta, i}(x, t, c)$$
non dipende arbitrariamente da tutti i 20.000 geni del genoma!
Dipende primariamente dallo stato dei suoi **Fattori di Trascrizione a monte** $\mathcal{N}_{\text{in}}(i)$, mediato dall'attività degli elementi regolatori cis (promotori ed enhancer).

Traduciamo questo vincolo biologico in un operatore di **Message Passing su Grafo Diretto**:
$$\begin{aligned}
\text{Messaggio: } & m_{j \to i} = A_{\text{eff}}(j \to i) \cdot W_{\text{msg}} h_j \\
\text{Aggregazione: } & m_i = \sum_{j \in \mathcal{N}_{\text{in}}(i)} m_{j \to i} \\
\text{Aggiornamento: } & h_i^{(l+1)} = h_i^{(l)} + \text{MLP}\big([h_i^{(l)}, m_i]\big) \\
\text{Readout: } & v_i = \text{Linear}(h_i^{(L)}) \in \mathbb{R}
\end{aligned}$$


---
## 2. Setup e Import


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import torch.nn as nn

from models.graph_vector_field import DirectedGeneConv, GraphVectorField
from data.synthetic_grn import SyntheticGRNSimulator
print('Modelli GNN caricati con successo!')


---
## 3. Lo Strato `DirectedGeneConv` e l'Aggregazione Sparsa

Osserviamo come `scatter_add_` aggrega i messaggi in parallelo su GPU per tutti i geni bersaglio.


In [ ]:
sim = SyntheticGRNSimulator(num_genes=8, density=0.3, seed=42)
conv = DirectedGeneConv(in_channels=16, out_channels=16)

B = 2  # batch di 2 cellule
h = torch.randn(B, sim.num_genes, 16)
edge_weights = torch.rand(B, sim.num_edges)

h_out = conv(h, sim.edge_index, edge_weights)
print(f'Shape input h: {h.shape}')
print(f'Shape output h_out: {h_out.shape}')
assert h_out.shape == h.shape, 'La forma deve essere preservata!'


---
## 4. La Rete Completa `GraphVectorField`

Combina l'encoder cellulare, la maschera di attività dinamica, $L$ strati di convoluzione orientata e il readout scalare.


In [ ]:
model = GraphVectorField(
    num_genes=sim.num_genes,
    edge_index=sim.edge_index,
    perturbation_dim=sim.num_genes,
    hidden_dim=32,
    num_layers=2,
    use_mask=True,
)

# Forward pass con un batch di test
x = torch.rand(B, sim.num_genes)
t = torch.full((B, 1), 0.3)
c = torch.zeros(B, sim.num_genes)
c[0, 1] = 1.0  # Cellula 0 ha knockout sul Gene 1

v_pred = model(x, t, perturbation=c)
print('Shape del campo vettoriale predetto v:', v_pred.shape)  # [B, G]
print('Velocità calcolate per la cellula 0:', v_pred[0].detach().numpy().round(3))


---
## 5. Ispezione della Maschera di Attività per Interpretazione Biologica

Il metodo `get_latest_mask` permette ai biologi computazionali di ispezionare quali archi sono stati ritenuti responsabili della risposta cellulare.


In [ ]:
mask_act = model.get_latest_mask(x, t, perturbation=c)
print(f'Pesi degli archi attivi durante il knockout (primi 5 archi): {mask_act[0, :5].detach().numpy().round(3)}')
